In [ ]:
# VAD + Diarisation tier writer for separated A/B files
# and diarisation transfer onto reference mixtures.

import os, re, traceback
from pathlib import Path
from typing import List, Tuple, Optional, Dict

HUGGINGFACE_TOKEN = "TOKEN_HERE"

SEPARATED_DIR = Path("/Users/moanason/Downloads/Data_REC")  # s01/s02/... with *_A/B_*_processed.wav
REFERENCE_DIR = Path("/Users/moanason/Downloads/Data_REF")  # rec_s##_NC#.wav

# to match separated (processed) files:
#   p142_s06_A_NC1_processed.wav
SEP_RX = re.compile(
    r"^p\d+_s(?P<sess>\d+)_"
    r"(?P<ab>[AB])_"
    r"NC(?P<nc>\d+)_processed\.wav$",
    re.IGNORECASE,
)

# to match reference mixtures:
#   rec_s06_NC1.wav
REF_RX = re.compile(r"^rec_s(?P<sess>\d+)_NC(?P<nc>\d+)\.wav$", re.IGNORECASE)

# Tier names
VAD_TIER_NAME = "VAD"               # non-speech / speech
DIAR_TIER_NAME = "Diarisation"      # for separated A/B files
REF_DIAR_A = "Diarisation_A"        # for reference mixture
REF_DIAR_B = "Diarisation_B"

# labels
SPEECH_LABEL = "speech"
NON_SPEECH_LABEL = "non-speech"

# to label diarisation on separated files:
#   "ab" -> "A" or "B"
#   "sess_ab" -> "s06_A" style
DIAR_LABEL_STYLE = "sess_ab"   # or "ab"

# knobs (same as VAD_TextGrid_librosa.py)
MIN_SEG_DUR = 0.002   # drop segments shorter than 20 ms after clipping
MERGE_TOL   = 0.002  # merge if gap/overlap <= 2 ms
WARN_TOL    = 0.001  # warn if |TG xmax - WAV dur| > 1 ms

from tqdm import tqdm
import numpy as np
import soundfile as sf
from praatio import tgio
from pyannote.audio import Pipeline

def wav_duration_sec(path: Path) -> float:
    with sf.SoundFile(str(path)) as f:
        return len(f) / float(f.samplerate)

def load_or_new_textgrid(wav_path: Path) -> tgio.Textgrid:
    """
    Load an existing .TextGrid next to wav_path, or return a new empty one.
    """
    tg_path = wav_path.with_suffix(".TextGrid")
    if tg_path.exists():
        try:
            return tgio.openTextgrid(str(tg_path), includeEmptyIntervals=True)
        except Exception:
            # corrupt or unreadable -> start fresh
            return tgio.Textgrid()
    return tgio.Textgrid()

def save_textgrid_for(path_wav: Path, tg: tgio.Textgrid, output_root: Optional[Path] = None) -> Path:
    if output_root is None:
        out_path = path_wav.with_suffix(".TextGrid")
    else:
        rel = path_wav.relative_to(SEPARATED_DIR if SEPARATED_DIR in path_wav.parents else REFERENCE_DIR)
        out_path = (output_root / rel).with_suffix(".TextGrid")
        out_path.parent.mkdir(parents=True, exist_ok=True)
    tg.save(str(out_path), minimumIntervalLength=0.0, outputFormat="textgrid")
    return out_path

def upsert_tier(tg: tgio.Textgrid, tier_name: str, entries: List[Tuple[float, float, str]], xmax: float):
    # Praatio wants tier bounds; use [0.0, xmax]
    new_tier = tgio.IntervalTier(tier_name, entries, 0.0, float(xmax))
    # remove if exists
    try:
        _ = tg.tierDict[tier_name]
        tg.deleteTier(tier_name)
    except KeyError:
        pass
    tg.addTier(new_tier)


def merge_intervals(intervals: List[Tuple[float, float]], tol: float = 0.0) -> List[Tuple[float, float]]:
    if not intervals:
        return []
    intervals = sorted((float(s), float(e)) for s, e in intervals)
    merged = []
    s0, e0 = intervals[0]
    for s, e in intervals[1:]:
        if s <= e0 + tol:
            e0 = max(e0, e)
        else:
            merged.append((s0, e0))
            s0, e0 = s, e
    merged.append((s0, e0))
    return merged

def clip_intervals(intervals: List[Tuple[float, float]], total_dur: float, min_len: float = 0.0) -> List[Tuple[float, float]]:
    out = []
    for s, e in intervals:
        s = max(0.0, float(s))
        e = min(float(total_dur), float(e))
        if e - s >= min_len:
            out.append((s, e))
    return out

def invert_intervals(speech: List[Tuple[float, float]], total_dur: float, min_len: float = 0.0) -> List[Tuple[float, float]]:
    out = []
    cursor = 0.0
    for s, e in speech:
        if s > cursor and (s - cursor) >= min_len:
            out.append((cursor, s))
        cursor = max(cursor, e)
    if total_dur - cursor >= min_len:
        out.append((cursor, total_dur))
    return out

def iter_separated_wavs(sep_root: Path):
    for sess in sorted(p for p in sep_root.iterdir() if p.is_dir()):
        for w in sess.glob("*.wav"):
            m = SEP_RX.match(w.name)
            if m:
                yield w, m.groupdict()

def iter_reference_wavs(ref_root: Path):
    for w in sorted(ref_root.glob("rec_s*_NC*.wav")):
        m = REF_RX.match(w.name)
        if m:
            yield w, m.groupdict()

# VAD runner
def run_vad(pipeline: Pipeline, wav_path: Path) -> List[Tuple[float, float]]:
    ann = pipeline(str(wav_path))
    spans = []
    for segment, _, label in ann.itertracks(yield_label=True):
        if str(label).upper() == "SPEECH":
            spans.append((float(segment.start), float(segment.end)))
    return spans

def diar_label(sess: str, ab: str) -> str:
    if DIAR_LABEL_STYLE == "ab":
        return ab
    return f"s{int(sess):02d}_{ab}"

# our main routines: process separated files and reference files

def process_separated_files(pipeline: Pipeline) -> Tuple[int, List[Tuple[str, str]]]:
    print("Scanning separated (A/B) files...")
    files = list(iter_separated_wavs(SEPARATED_DIR))
    print(f"  {len(files)} separated files found under {SEPARATED_DIR}.")
    errors = []
    updated = 0

    for wav_path, meta in tqdm(files, desc="Separated -> VAD & Diarisation"):
        try:
            sess = meta["sess"]
            ab   = meta["ab"].upper()
            dur  = wav_duration_sec(wav_path)
            # VAD
            raw_speech = run_vad(pipeline, wav_path)
            speech = merge_intervals(clip_intervals(raw_speech, dur, min_len=0.0), tol=MERGE_TOL)
            nonspeech = invert_intervals(speech, dur, min_len=MIN_SEG_DUR)

            # build entries
            vad_entries = []
            for s, e in nonspeech: vad_entries.append((s, e, NON_SPEECH_LABEL))
            for s, e in speech:    vad_entries.append((s, e, SPEECH_LABEL))
            vad_entries.sort(key=lambda x: x[0])

            diar_entries = [(s, e, diar_label(sess, ab)) for s, e in speech]

            # load or create TG, upsert tiers, save
            tg = load_or_new_textgrid(wav_path)
            upsert_tier(tg, VAD_TIER_NAME, vad_entries, dur)
            upsert_tier(tg, DIAR_TIER_NAME, diar_entries, dur)

            out_path = save_textgrid_for(wav_path, tg, output_root=None)
            updated += 1

            if abs(dur - dur) > WARN_TOL:  # no-op guard; keeps symmetry with earlier check template
                pass

        except Exception as e:
            errors.append((str(wav_path), traceback.format_exc().splitlines()[-1]))

    return updated, errors


def process_reference_files(pipeline: Pipeline) -> Tuple[int, List[Tuple[str, str]]]:
    """
    For each rec_s##_NC#.wav, run VAD on the aligned A/B _processed.wav in the matching session
    and write two tiers on the reference TG: Diarisation_A and Diarisation_B.
    """
    print("Scanning reference (mixture) files...")
    refs = list(iter_reference_wavs(REFERENCE_DIR))
    print(f"  {len(refs)} reference files found under {REFERENCE_DIR}.")
    errors = []
    updated = 0

    for ref_path, meta in tqdm(refs, desc="Reference -> Diarisation tiers from A/B"):
        try:
            sess = meta["sess"]
            nc   = meta["nc"]
            dur  = wav_duration_sec(ref_path)

            # find the corresponding A/B processed files
            sess_dir = SEPARATED_DIR / f"s{int(sess):02d}"
            cand_A = list(sess_dir.glob(f"p*_s{int(sess):02d}_A_NC{int(nc)}_processed.wav"))
            cand_B = list(sess_dir.glob(f"p*_s{int(sess):02d}_B_NC{int(nc)}_processed.wav"))
            if not cand_A or not cand_B:
                raise RuntimeError(f"Missing aligned A/B processed wavs for s{int(sess):02d} NC{nc}")

            A_path = cand_A[0]
            B_path = cand_B[0]

            # VAD on the *aligned* A/B processed files (times now match the reference)
            speech_A = merge_intervals(clip_intervals(run_vad(pipeline, A_path), dur, 0.0), tol=MERGE_TOL)
            speech_B = merge_intervals(clip_intervals(run_vad(pipeline, B_path), dur, 0.0), tol=MERGE_TOL)

            # build entries (only speech intervals in each tier)
            diar_A_entries = [(s, e, diar_label(sess, "A")) for s, e in speech_A]
            diar_B_entries = [(s, e, diar_label(sess, "B")) for s, e in speech_B]

            tg = load_or_new_textgrid(ref_path)
            upsert_tier(tg, REF_DIAR_A, diar_A_entries, dur)
            upsert_tier(tg, REF_DIAR_B, diar_B_entries, dur)

            save_textgrid_for(ref_path, tg, output_root=None)
            updated += 1

        except Exception as e:
            errors.append((str(ref_path), traceback.format_exc().splitlines()[-1]))

    return updated, errors


if __name__ == "__main__":
    try:
        pipeline = Pipeline.from_pretrained(
            "pyannote/voice-activity-detection",
            use_auth_token=HUGGINGFACE_TOKEN
        )
        print("pyannote VAD pipeline loaded.")
    except Exception as e:
        raise RuntimeError(
            "Failed to load pyannote pipeline. If you see a 403, enable "
            "'Access to gated models' for this token on Hugging Face."
        ) from e

    upd_sep, err_sep = process_separated_files(pipeline)
    print(f"Separated done. Updated {upd_sep} TextGrids.")
    if err_sep:
        print("Errors (first 5):")
        for p, msg in err_sep[:5]:
            print(f"  {p}: {msg}")

    upd_ref, err_ref = process_reference_files(pipeline)
    print(f"Reference done. Created/updated {upd_ref} TextGrids.")
    if err_ref:
        print("Reference errors (first 5):")
        for p, msg in err_ref[:5]:
            print(f"  {p}: {msg}")


Lightning automatically upgraded your loaded checkpoint from v1.1.3 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/torch/pyannote/models--pyannote--segmentation/snapshots/059e96f964841d40f1a5e755bb7223f76666bba4/pytorch_model.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.4.0. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.7.1, yours is 2.6.0. Bad things might happen unless you revert torch to 1.x.
pyannote VAD pipeline loaded.
Scanning separated (A/B) files...
  12 separated files found under /Users/moanason/Downloads/Data_REC.


Separated -> VAD & Diarisation: 100%|██████████| 12/12 [06:51<00:00, 34.29s/it]


Separated done. Updated 12 TextGrids.
Scanning reference (mixture) files...
  6 reference files found under /Users/moanason/Downloads/Data_REF.


Reference -> Diarisation tiers from A/B: 100%|██████████| 6/6 [06:46<00:00, 67.75s/it]

Reference done. Created/updated 6 TextGrids.
